## Creating Medallion Layer Schemas

In this step, creating separate schemas for the Medallion Architecture layers: Bronze, Silver, and Gold.

This would help organize tables cleanly, keeps raw/clean/aggregated data separated, and makes the pipeline easy to maintain.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecom_bronze;
CREATE SCHEMA IF NOT EXISTS ecom_silver;
CREATE SCHEMA IF NOT EXISTS ecom_gold;

## Load Source Dataset

Loading the source dataset from the existing table default.ecommerce_transactions.

Also previewing a few records to understand the data structure and confirming the available columns.

In [0]:
events = spark.table("default.ecommerce_transactions")
display(events.limit(10))

Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
1,Ava Hall,63,Mexico,Clothing,780.69,Debit Card,2023-04-14
2,Sophia Hall,59,India,Beauty,738.56,PayPal,2023-07-30
3,Elijah Thompson,26,France,Books,178.34,Credit Card,2023-09-17
4,Elijah White,43,Mexico,Sports,401.09,UPI,2023-06-21
5,Ava Harris,48,Germany,Beauty,594.83,Net Banking,2024-10-29
6,Elijah Harris,51,India,Toys,966.5,Cash on Delivery,2025-01-18
7,Oliver Clark,27,Germany,Home & Kitchen,341.73,Credit Card,2024-03-13
8,Olivia Allen,46,Canada,Home & Kitchen,11.33,Debit Card,2024-01-04
9,Liam Harris,54,France,Beauty,279.43,Cash on Delivery,2023-12-06
10,Liam Allen,60,Canada,Beauty,223.9,Cash on Delivery,2023-08-07


## Inspecting Schema and Row Count

In this step, validating the source dataset by checking:

Total number of rows, Column names and Data types (schema)

In [0]:
print(events.count())
print(events.columns)
events.printSchema()

50000
['Transaction_ID', 'User_Name', 'Age', 'Country', 'Product_Category', 'Purchase_Amount', 'Payment_Method', 'Transaction_Date']
root
 |-- Transaction_ID: long (nullable = true)
 |-- User_Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Purchase_Amount: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Transaction_Date: date (nullable = true)



## Bronze Layer: Raw Ingestion

This step creates the Bronze layer table, which stores the raw data exactly as received from the source.

We also add ingestion metadata columns like ingestion timestamp and source name for traceability and debugging.

In [0]:
from pyspark.sql.functions import current_timestamp, lit, to_date
events = spark.table("default.ecommerce_transactions")

bronze_df = (
    events.withColumn("ingest_ts", current_timestamp())
    .withColumn("source_name", lit("ecommerce_transactions"))
    .withColumn("ingest_date", to_date(current_timestamp()))
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable("ecom_bronze.transactions_bronze")

In [0]:
bronze = spark.table("ecom_bronze.transactions_bronze")
print("bronze count: ", bronze.count())
display(bronze.limit(10))

bronze count:  50000


Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date,ingest_ts,source_name,ingest_date
1,Ava Hall,63,Mexico,Clothing,780.69,Debit Card,2023-04-14,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
2,Sophia Hall,59,India,Beauty,738.56,PayPal,2023-07-30,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
3,Elijah Thompson,26,France,Books,178.34,Credit Card,2023-09-17,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
4,Elijah White,43,Mexico,Sports,401.09,UPI,2023-06-21,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
5,Ava Harris,48,Germany,Beauty,594.83,Net Banking,2024-10-29,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
6,Elijah Harris,51,India,Toys,966.5,Cash on Delivery,2025-01-18,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
7,Oliver Clark,27,Germany,Home & Kitchen,341.73,Credit Card,2024-03-13,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
8,Olivia Allen,46,Canada,Home & Kitchen,11.33,Debit Card,2024-01-04,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
9,Liam Harris,54,France,Beauty,279.43,Cash on Delivery,2023-12-06,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16
10,Liam Allen,60,Canada,Beauty,223.9,Cash on Delivery,2023-08-07,2026-02-16T09:37:59.588Z,ecommerce_transactions,2026-02-16


## Data Quality Checks on Bronze

In this step, it is important to perform basic data quality checks on the Bronze layer.

Verifying row counts, uniqueness of Transaction_ID, null checks, and invalid purchase amount checks.

This helps to ensure we understand the data quality before cleaning it in Silver.

In [0]:
from pyspark.sql.functions import count, countDistinct, col, sum
bronze = spark.table("ecom_bronze.transactions_bronze")
display(
    bronze.select(
        count("*").alias("rows"),
        countDistinct("Transaction_ID").alias("distinct_txn_id"),
        sum((col("Transaction_ID").isNull()).cast("int")).alias("null_txn_id "),
        sum((col("Purchase_Amount").isNull()).cast("int")).alias("null_purchase_amt"),
        sum((col("Transaction_Date").isNull()).cast("int")).alias("null_transaction_date"),
        sum((col("Purchase_Amount") <= 0).cast("int")).alias("invalid_purchase_amount")
    )
)

rows,distinct_txn_id,null_txn_id,null_purchase_amt,null_transaction_date,invalid_purchase_amount
50000,50000,0,0,0,0


## SILVER Layer: Clean and Standardize Data

This step creates the Silver layer table by cleaning and standardizing the Bronze data.

Applying basic validation rules, trim string columns, add reusable derived columns (year/month), and remove duplicates to produce a trusted dataset for analytics.

In [0]:
import pyspark.sql.functions as F

bronze = spark.table("ecom_bronze.transactions_bronze")
silver_df = (
    bronze.filter(F.col("Purchase_Amount") > 0)
    .withColumn("UserName", F.trim(F.col("User_Name")))
    .withColumn("Country", F.trim(F.col("Country")))
    .withColumn("ProductCategory", F.trim(F.col("Product_Category")))
    .withColumn("PaymentMethod", F.trim(F.col("Payment_Method")))
    .withColumn("transaction_year", F.year("Transaction_Date"))
    .withColumn("transaction_month", F.month("Transaction_Date"))
    .dropDuplicates(["Transaction_ID"])
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("ecom_silver.transactions_silver")

## Validate Silver Table

After creating the Silver table, validating the transformation by checking row count and previewing key columns.

This ensures our cleaned dataset is ready for Gold-level aggregations.



In [0]:
silver = spark.table("ecom_silver.transactions_silver")
print("Silver count: ", silver.count())
display(silver.select("Transaction_ID", "UserName","Purchase_Amount", "ProductCategory", "Transaction_Date", "transaction_year", "transaction_month").limit(5))

Silver count:  50000


Transaction_ID,UserName,Purchase_Amount,ProductCategory,Transaction_Date,transaction_year,transaction_month
28,Ava Thompson,454.08,Grocery,2023-09-15,2023,9
29,Noah Harris,911.02,Beauty,2023-03-28,2023,3
30,Emma White,381.67,Home & Kitchen,2023-05-16,2023,5
33,Olivia Hall,790.2,Electronics,2023-11-19,2023,11
93,Liam Thompson,341.89,Electronics,2024-03-28,2024,3


## GOLD Layer: Daily Revenue

This Gold table aggregates Silver level data.

Calcuating key metrics like total revenue, total transactions, and unique users per day.

In [0]:
from pyspark.sql import functions as F

silver = spark.table("ecom_silver.transactions_silver")

gold_daily_revenue = (
    silver.groupBy("Transaction_Date")
    .agg(
        F.sum("Purchase_Amount").alias("Daily_Revenue"),
        F.count("*").alias("total_transactions"),
        F.countDistinct("User_Name").alias("unique_users")
    )
    .orderBy("Transaction_Date")
)

gold_daily_revenue.write.format("delta").mode("overwrite").saveAsTable("ecom_gold.daily_revenue")

print("daily_revenue : ", spark.table("ecom_gold.daily_revenue").count())
display(spark.table("ecom_gold.daily_revenue").limit(5))

daily_revenue :  731


Transaction_Date,Daily_Revenue,total_transactions,unique_users
2023-03-09,34211.92,64,44
2023-03-10,35190.87,75,54
2023-03-11,33670.82000000001,72,51
2023-03-12,40101.44,74,48
2023-03-13,32031.160000000007,68,50


## Gold Layer: Revenue by Product Category

In [0]:
gold_category_revenue = (
    silver.groupBy("Product_Category")
    .agg(
        F.sum("Purchase_Amount").alias("total_revenue"),
        F.count("*").alias("total_transactions"),
        F.countDistinct("User_Name").alias("unique_users")
    )
    .orderBy(F.desc("total_revenue"))
)

gold_category_revenue.write.format("delta").mode("overwrite").saveAsTable("ecom_gold.category_revenue")

print("category_revenue : ", spark.table("ecom_gold.category_revenue").count())
display(spark.table("ecom_gold.category_revenue").limit(5))

category_revenue :  8


Product_Category,total_revenue,total_transactions,unique_users
Sports,3195335.8999999994,6312,100
Toys,3185652.360000002,6392,100
Books,3181897.300000001,6253,100
Clothing,3171225.9599999986,6224,100
Electronics,3133965.040000001,6320,100


## Gold Layer: Revenue by Country

In [0]:
gold_country_revenue = (
    silver.groupBy("Country")
    .agg(
        F.sum("Purchase_Amount").alias("total_revenue"),
        F.count("*").alias("total_transactions"),
        F.countDistinct("User_Name").alias("unique_users")        
    )
    .orderBy(F.desc("total_revenue"))
)

gold_category_revenue.write.format("delta").mode("overwrite").saveAsTable("ecom_gold.country_revenue")
print("country_revenue : ", spark.table("ecom_gold.country_revenue").count())
display(spark.table("ecom_gold.country_revenue").limit(5))

country_revenue :  8


Product_Category,total_revenue,total_transactions,unique_users
Sports,3195335.8999999994,6312,100
Toys,3185652.360000002,6392,100
Books,3181897.300000001,6253,100
Clothing,3171225.9599999986,6224,100
Electronics,3133965.040000001,6320,100
